In [ ]:
# ============================================================
# ROUNDOFF NOISE AT THE OUTPUT OF A FIRST-ORDER IIR FILTER
# ============================================================
#
# This notebook demonstrates the impact of finite word length
# on the output roundoff-noise power of the first-order IIR system
#
#                   y[n] = alpha y[n-1] + x[n]
#
# under the white-noise model of quantization.
#
# ------------------------------------------------------------
# THEORY
# ------------------------------------------------------------
#
# For a first-order IIR filter in direct form I, the noise model
# replaces the quantizers by additive white-noise sources.
#
# For this simple system:
#
#   - M = 0
#   - N = 1
#   - M + N + 1 = 2
#
# Hence, under fixed-point rounding with K fractional bits,
#
#                   Delta = 2^(-K)
#
# the variance of one rounding-noise source is
#
#                   sigma_q^2 = Delta^2 / 12
#
# and the equivalent input-noise variance becomes
#
#                   sigma_e^2 = 2 * Delta^2 / 12.
#
# The transfer function from the equivalent noise source e[n]
# to the output noise y_e[n] is
#
#                   H(z) = 1 / (1 - alpha z^(-1))
#
# with impulse response
#
#                   h[n] = alpha^n u[n].
#
# Therefore,
#
#                   sum |h[n]|^2 = 1 / (1 - |alpha|^2)
#
# for |alpha| < 1, and the theoretical output-noise variance is
#
#                   sigma_y_e^2
#
#                   = 2 * 2^(-2K) / 12
#                     * 1 / (1 - |alpha|^2).
#
#
# ------------------------------------------------------------
# IMPORTANT INTERPRETATION OF THE ALPHA DEPENDENCE
# ------------------------------------------------------------
#
# For a fixed value of K,
#
#                   2 * 2^(-2K) / 12
#
# is constant.
#
# Therefore, the variation of the output-noise variance with alpha
# is determined entirely by the noise-gain factor
#
#                   G_N = 1 / (1 - |alpha|^2).
#
# As |alpha| approaches 1,
#
#                   1 - |alpha|^2 -> 0
#
# and therefore
#
#                   G_N -> infinity
#
# and
#
#                   sigma_y_e^2 -> infinity.
#
# Thus, a pole close to the unit circle greatly amplifies the
# roundoff noise.
#
# Since the expression depends on |alpha|^2, the variance curve
# is symmetric with respect to alpha = 0.
#
# For example, alpha = +0.9 and alpha = -0.9 produce the same
# theoretical output-noise variance.
#
#
# ------------------------------------------------------------
# HOW TO USE THE NOTEBOOK
# ------------------------------------------------------------
#
# 1. Choose the quantity to display:
#
#       - Output noise variance vs alpha
#       - Output noise variance vs K
#       - Impulse response
#       - Squared magnitude response
#
# 2. Change:
#
#       - alpha
#       - K (fractional bits)
#       - simulation length
#
# 3. Observe:
#
#       - how the output-noise variance decreases as K increases,
#       - how the output-noise variance grows as |alpha| approaches 1,
#       - how the impulse response becomes longer as |alpha| increases,
#       - how the frequency response explains the noise amplification.
#
#
# ------------------------------------------------------------
# IMPORTANT MODELING ASSUMPTIONS
# ------------------------------------------------------------
#
# This notebook uses the following simplifying assumptions:
#
#   1. Fixed-point arithmetic is used.
#
#   2. Quantization is performed by rounding.
#
#   3. The two quantization-noise sources are modeled as
#      statistically independent white-noise sources.
#
#   4. The displayed theoretical formula is valid in the stable
#      region |alpha| < 1.
#
#   5. The simulation uses two independent uniform-noise sequences
#      in [-Delta/2, Delta/2], adds them, and passes the result
#      through the first-order IIR recursion.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, Layout, interactive_output
from IPython.display import display


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def quantization_parameters(K):

    Delta = 2.0**(-K)

    sigma_q2 = Delta**2 / 12.0

    sigma_e2 = 2.0 * sigma_q2

    return Delta, sigma_q2, sigma_e2


def theoretical_output_noise_variance(alpha, K):

    _, _, sigma_e2 = quantization_parameters(K)

    return sigma_e2 / (1.0 - np.abs(alpha)**2)


def impulse_response(alpha, L=60):

    n = np.arange(L)

    h = alpha**n

    return n, h


def squared_magnitude_response(alpha, omega):

    H = 1.0 / (1.0 - alpha * np.exp(-1j * omega))

    return np.abs(H)**2


def simulate_output_noise(alpha, K, N, discard_fraction=0.20, seed=0):

    rng = np.random.default_rng(seed)

    Delta, sigma_q2, sigma_e2 = quantization_parameters(K)

    e1 = rng.uniform(-Delta / 2.0, Delta / 2.0, N)

    e2 = rng.uniform(-Delta / 2.0, Delta / 2.0, N)

    e = e1 + e2

    y = np.zeros(N)

    y[0] = e[0]

    for n in range(1, N):

        y[n] = alpha * y[n - 1] + e[n]

    discard = int(discard_fraction * N)

    e_ss = e[discard:]

    y_ss = y[discard:]

    measured_input_variance = np.var(e_ss)

    measured_output_variance = np.var(y_ss)

    return e, y, measured_input_variance, measured_output_variance, sigma_e2


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.rn-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.rn-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.rn-interpretation {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7d8c9;
    border-left: 6px solid #3c8a4e;
    background: #f7fbf7;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.rn-interpretation-title {
    font-size: 13px;
    font-weight: bold;
    color: #245c31;
    margin-bottom: 5px;
}

.rn-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.rn-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.rn-info {
    font-size: 13px;
    line-height: 1.52;
}

.rn-label {
    display: inline-block;
    min-width: 245px;
    font-weight: bold;
}

.rn-value {
    font-size: 15px;
    font-weight: bold;
}

.rn-good {
    color: #176b34;
    font-weight: bold;
}

.rn-warn {
    color: #a85b00;
    font-weight: bold;
}

.rn-assumption {
    margin-top: 6px;
    padding: 6px 8px;
    border-radius: 6px;
    background: #f2f4f7;
    font-size: 12px;
    line-height: 1.35;
}

.rn-controls {
    overflow-x: visible !important;
    overflow-y: visible !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="rn-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Roundoff Noise at the Output of a First-Order IIR Filter
    </div>

</div>
""")


# ------------------------------------------------------------
# Description and visible interpretation
# ------------------------------------------------------------

description_html = HTML("""
<div class="rn-root">

    <div class="rn-description">

        This notebook studies the first-order IIR system
        <b>y[n] = α y[n−1] + x[n]</b> under the standard white-noise
        model of quantization.<br><br>

        The displayed theory assumes <b>fixed-point rounding</b>,
        <b>two independent quantization-noise sources</b> in direct form I,
        and the <b>stable operating region |α| &lt; 1</b>.<br><br>

        The theoretical output-noise variance is

        <div style="text-align:center; margin:7px 0;">
            <b>
            σ²<sub>y<sub>e</sub></sub>
            =
            2·2<sup>−2K</sup>/12 · 1/(1−|α|²).
            </b>
        </div>

        The notebook also computes a Monte Carlo simulation using two
        independent uniform noise sources in <b>[−Δ/2, Δ/2]</b>.

    </div>


    <div class="rn-interpretation">

        <div class="rn-interpretation-title">
            Interpretation of the variance-versus-α plot
        </div>

        For a fixed value of <b>K</b>, the factor
        <b>2·2<sup>−2K</sup>/12</b> is constant. Therefore, the variation
        of the output-noise variance with <b>α</b> is determined entirely
        by the noise gain

        <div style="text-align:center; margin:7px 0;">
            <b>
            G<sub>N</sub> = 1/(1−|α|²).
            </b>
        </div>

        As <b>|α| approaches 1</b>, the denominator
        <b>1−|α|² approaches zero</b>, so the noise gain and the
        output-noise variance increase very rapidly.<br><br>

        Therefore, <b>a pole located close to the unit circle strongly
        amplifies roundoff noise</b>.<br><br>

        Since the expression depends on <b>|α|²</b>, the curve is symmetric
        with respect to <b>α = 0</b>. Thus, for example,
        <b>α = +0.9</b> and <b>α = −0.9</b> produce the same theoretical
        output-noise variance.

    </div>

</div>
""")


# ------------------------------------------------------------
# Dynamic summary
# ------------------------------------------------------------

summary_html = HTML()

summary_html.layout = Layout(
    width='610px',
    min_width='610px',
    overflow='visible'
)


# ------------------------------------------------------------
# Main interactive function
# ------------------------------------------------------------

def plot_roundoff_noise(display_mode='Output noise variance vs alpha', alpha=0.85, K=12, N=4096):

    Delta, sigma_q2, sigma_e2 = quantization_parameters(K)

    sigma_y2_theoretical = theoretical_output_noise_variance(alpha, K)

    e, y, measured_input_variance, measured_output_variance, sigma_e2_model = simulate_output_noise(alpha, K, N)

    noise_gain = 1.0 / (1.0 - np.abs(alpha)**2)

    relative_error = 100.0 * np.abs(measured_output_variance - sigma_y2_theoretical) / sigma_y2_theoretical

    if relative_error < 10.0:

        agreement_text = "<span class='rn-good'>Theory and simulation are in close agreement.</span>"

    else:

        agreement_text = "<span class='rn-warn'>Theory and simulation show noticeable finite-length discrepancy.</span>"


    summary_html.value = f"""
    <div class="rn-box">

        <div class="rn-title">
            Current Numerical Summary
        </div>

        <div class="rn-info">

            <span class="rn-label">Pole coefficient</span>
            α = <span class="rn-value">{alpha:.4f}</span>
            <br>

            <span class="rn-label">Fractional bits</span>
            K = <span class="rn-value">{K}</span>
            <br>

            <span class="rn-label">Simulation length</span>
            N = {N}
            <br>

            <span class="rn-label">Quantization step</span>
            Δ = 2<sup>-{K}</sup> = {Delta:.8e}
            <br>

            <span class="rn-label">Variance of one quantizer</span>
            σ²<sub>q</sub> = Δ² / 12 = {sigma_q2:.8e}
            <br>

            <span class="rn-label">Equivalent input-noise variance</span>
            σ²<sub>e</sub> = 2Δ² / 12 = {sigma_e2:.8e}
            <br>

            <span class="rn-label">Noise gain</span>
            1 / (1 − |α|²) = <span class="rn-value">{noise_gain:.8f}</span>
            <br>

            <span class="rn-label">Theoretical output-noise variance</span>
            <span class="rn-value">
            σ²<sub>y<sub>e</sub></sub> = {sigma_y2_theoretical:.8e}
            </span>
            <br>

            <span class="rn-label">Measured input-noise variance</span>
            {measured_input_variance:.8e}
            <br>

            <span class="rn-label">Measured output-noise variance</span>
            <span class="rn-value">{measured_output_variance:.8e}</span>
            <br>

            <span class="rn-label">Relative difference</span>
            {relative_error:.3f} %
            <br>

            <span class="rn-label">Agreement</span>
            {agreement_text}

        </div>

        <div class="rn-assumption">
            <b>Modeling assumptions:</b>
            fixed-point rounding, direct-form I with two independent
            quantization-noise sources, white-noise approximation, and
            stable operation with |α| &lt; 1.
        </div>

    </div>
    """


    fig, ax = plt.subplots(
        figsize=(11.4, 4.8)
    )


    # ========================================================
    # OUTPUT NOISE VARIANCE VS ALPHA
    # ========================================================

    if display_mode == 'Output noise variance vs alpha':

        alpha_grid = np.linspace(-0.98, 0.98, 600)

        sigma_grid = theoretical_output_noise_variance(alpha_grid, K)

        ax.plot(alpha_grid, sigma_grid, linewidth=2.0, label='Theoretical output-noise variance')

        ax.plot(alpha, sigma_y2_theoretical, 'o', markersize=8, label='Current operating point')

        ax.axvline(alpha, linewidth=1.0, linestyle='--', label='Current α')

        ax.set_xlabel('Pole coefficient α')

        ax.set_ylabel(r'$\sigma_{y_e}^2$')

        ax.set_title('Output Roundoff-Noise Variance versus Pole Coefficient', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.6)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=3, fontsize=9, frameon=False)


    # ========================================================
    # OUTPUT NOISE VARIANCE VS K
    # ========================================================

    elif display_mode == 'Output noise variance vs K':

        K_grid = np.arange(2, 17)

        sigma_grid = np.array([theoretical_output_noise_variance(alpha, k) for k in K_grid])

        ax.plot(K_grid, sigma_grid, 'o-', linewidth=1.8, markersize=5, label='Theoretical output-noise variance')

        ax.plot(K, sigma_y2_theoretical, 'o', markersize=9, label='Current operating point')

        ax.set_yscale('log')

        ax.set_xlabel('Fractional bits K')

        ax.set_ylabel(r'$\sigma_{y_e}^2$')

        ax.set_title('Output Roundoff-Noise Variance versus Word Length', fontsize=12)

        ax.set_xticks(K_grid)

        ax.grid(True, linestyle=':', alpha=0.6)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9, frameon=False)


    # ========================================================
    # IMPULSE RESPONSE
    # ========================================================

    elif display_mode == 'Impulse response':

        n_h, h = impulse_response(alpha, L=60)

        markerline, stemlines, baseline = ax.stem(n_h, h, label='Impulse response h[n]', basefmt=' ')

        plt.setp(stemlines, linewidth=1.4)

        plt.setp(markerline, markersize=5)

        ax.set_xlabel('Sample index n')

        ax.set_ylabel('h[n]')

        ax.set_title('Impulse Response of the Noise-Transfer Filter', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.6)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, fontsize=9, frameon=False)


    # ========================================================
    # SQUARED MAGNITUDE RESPONSE
    # ========================================================

    else:

        omega = np.linspace(0.0, np.pi, 1200)

        mag2 = squared_magnitude_response(alpha, omega)

        ax.plot(omega, mag2, linewidth=2.0, label=r'$|H(e^{j\omega})|^2$')

        ax.set_xlim(0.0, np.pi)

        ax.set_xticks([0.0, np.pi / 4.0, np.pi / 2.0, 3.0 * np.pi / 4.0, np.pi])

        ax.set_xticklabels(['0', 'π/4', 'π/2', '3π/4', 'π'])

        ax.set_xlabel('Frequency ω')

        ax.set_ylabel(r'$|H(e^{j\omega})|^2$')

        ax.set_title('Squared Magnitude Response of the Noise-Transfer Filter', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.6)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, fontsize=9, frameon=False)


    plt.subplots_adjust(left=0.08, right=0.98, top=0.90, bottom=0.25)

    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(
    width='300px'
)


slider_style = {
    'description_width': '115px'
}


display_mode_selector = RadioButtons(
    options=[
        'Output noise variance vs alpha',
        'Output noise variance vs K',
        'Impulse response',
        'Squared magnitude response'
    ],
    value='Output noise variance vs alpha',
    description='Display:',
    style={'description_width': '65px'},
    layout=Layout(width='300px')
)


alpha_slider = FloatSlider(
    value=0.85,
    min=-0.98,
    max=0.98,
    step=0.01,
    description='alpha:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


K_slider = IntSlider(
    value=12,
    min=2,
    max=16,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


N_slider = IntSlider(
    value=4096,
    min=1024,
    max=16384,
    step=1024,
    description='Samples N:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls_box = VBox(
    [
        HTML("<div class='rn-title'>Controls</div>"),
        display_mode_selector,
        alpha_slider,
        K_slider,
        N_slider
    ],
    layout=Layout(
        width='340px',
        min_width='340px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible'
    )
)

controls_box.add_class('rn-controls')


# ------------------------------------------------------------
# Summary + controls
# ------------------------------------------------------------

top_row = HBox(
    [
        summary_html,
        controls_box
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Interactive output
# ------------------------------------------------------------

interactive_plot = interactive_output(
    plot_roundoff_noise,
    {
        'display_mode': display_mode_selector,
        'alpha': alpha_slider,
        'K': K_slider,
        'N': N_slider
    }
)


interactive_plot.layout = Layout(
    width='auto',
    overflow='visible'
)


# ------------------------------------------------------------
# Final layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        interactive_plot
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)